### Variational Fast Forwarding

VFF (variational fast forwarding) as introduced in https://www.nature.com/articles/s41534-020-00302-0, is a variational quantum algorithm to approximate a given unitary U (up to a global phase) as V^dagger D V where V is a variational ansatz and D a variational diagonal block. It is often used to approximate exp(-iHt) for small t ( which we name t0), such that this evolution for larger t (named t*) ("fast forwarding") can be approximated with V^dagger D^(t*/t0) V  

In [ ]:
import numpy as np
from openfermion import hamiltonians
from qarp.operators.compat import from_openfermion

from qarp.algorithms import VFF
from qarp.blocks import HnBlock, HEABlock, CompositeBlock, SynthesizedTimeEvolutionBlock, SPABlock
from qarp.operators import JordanWigner
from qarp.optimizers import AdamOptimizer
import qarpx as qx

We start with a simple example for approximately diagonalising a unitary U

In [ ]:
%load_ext autoreload
%autoreload 2

U =  HnBlock(n_qubits=3)
hea = HEABlock(n_qubits=3, n_layers=4, real=True, linear=False, circular=True, use_cz=True).build()
opt = AdamOptimizer({"maxiter":200, "lr":0.05})
vff = VFF(U, hea, optimizer=opt).build()
res = vff.run()



In [ ]:
print("Target Unitary")
U_mat = np.array(qx.QarpSimulator().unitary_matrix(U.flatten(), U.n_qubits))
print(U_mat)

print("Learnt Unitary")
parameter_map = dict(zip(vff.ansatz_block.symbols+ vff.D.symbols, res.x))

V = CompositeBlock([vff.ansatz_block.dagger(), vff.D.dagger(), vff.ansatz_block]).build()
V_res = V.set_symbols(parameter_map)
V_mat = np.array(qx.QarpSimulator().unitary_matrix(V_res.flatten(), V_res.n_qubits))

phase_approx = np.mean(np.divide(U_mat, V_mat))
V_mat = V_mat * phase_approx
print(V_mat)

print("Error")
print(np.mean(np.abs(U_mat-V_mat)))

We can also carry this out for time evolution exp(-iHt)

In [ ]:
t0 = 1
HH=hamiltonians.fermi_hubbard(
    2,
    1,
    -0.9,
    0.05,
    chemical_potential=0.1,
    magnetic_field=0.0,
    periodic=True,
    spinless=True,
    particle_hole_symmetry=False
)
HH = from_openfermion(HH)  # native boundary
qop = JordanWigner().encode_operator(HH)
hea = SPABlock(n_qubits=3, n_layers=4, real=False, linear=False, circular=True).build()
opt = AdamOptimizer({"maxiter":200, "lr":0.05})
vff = VFF(qop, hea, t_time=t0, optimizer = opt).build()
res = vff.run()


In [ ]:
print("Target Unitary")

_U_block = SynthesizedTimeEvolutionBlock(qop, 3, -1).build()
U_mat = np.array(qx.QarpSimulator().unitary_matrix(_U_block.flatten(), _U_block.n_qubits))
print(U_mat)

print("Learnt Unitary")

parameter_map = dict(zip(vff.ansatz_block.symbols+ vff.D.symbols, res.x))

V = CompositeBlock([vff.ansatz_block.dagger(), vff.D.dagger(), vff.ansatz_block]).build()
V_res = V.set_symbols(parameter_map)
V_mat = np.array(qx.QarpSimulator().unitary_matrix(V_res.flatten(), V_res.n_qubits))

phase_approx = U_mat[0,0]/V_mat[0,0]
V_mat = V_mat * phase_approx

print(V_mat)

print("Error")

print(np.mean(np.abs(U_mat-V_mat)))

The "fast forwward" to t=T of this time evolution can then be found with the following:

In [ ]:
T = 2
print("Target Unitary")
U_fastforward = SynthesizedTimeEvolutionBlock(qop, 3, -T).build()
U_fastforward_mat = np.array(qx.QarpSimulator().unitary_matrix(U_fastforward.flatten(), U_fastforward.n_qubits))
print(U_fastforward_mat)

print("Learnt Unitary")
parameter_map = dict(zip(vff.ansatz_block.symbols + vff.D.symbols, np.concatenate((res.x[0:len(vff.ansatz_block.symbols)], T/t0*res.x[len(vff.ansatz_block.symbols):]))))

V_fastforward = V.set_symbols(parameter_map)
V_fastforward_mat = np.array(qx.QarpSimulator().unitary_matrix(V_fastforward.flatten(), V_fastforward.n_qubits))
V_fastforward_mat = V_fastforward_mat * phase_approx

print(V_fastforward_mat)

print("Error")

print(np.mean(np.abs(U_fastforward_mat-V_fastforward_mat)))
